# Thư viện

In [28]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.datasets import fetch_20newsgroups
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

# Data

In [29]:
# 1. Tải dữ liệu 20 Newsgroups (chỉ lấy 4 chủ đề để demo chạy cho nhanh)
categories = ['sci.space', 'rec.autos', 'comp.graphics', 'sci.med']
print("Đang tải dữ liệu...")
train_data = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)

# 2. Tạo Input Vector X bằng TF-IDF
vectorizer = TfidfVectorizer(max_features=3000, stop_words='english')

X_train = vectorizer.fit_transform(train_data.data).toarray()
y_train = train_data.target


Đang tải dữ liệu...


# Model

In [30]:
# 1. Chuyển đổi dữ liệu Numpy array sang PyTorch Tensors
X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.long)


# 2. Định nghĩa Kiến trúc Mạng Nơ-ron (FNN)
class model(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(model, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, num_classes)

    def forward(self, a0, training_mode=True):
        z1 = self.layer1(a0)
        a1 = F.relu(z1)
        z2 = self.layer2(a1)

        if training_mode:
            return z2
        else:
            a2 = F.softmax(z2, dim=1)
            y_hat = a2
            return y_hat

# 3. Khởi tạo mô hình
model = model(input_size=3000, hidden_size=128, num_classes=4)

# Train

In [31]:
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

EPOCHS = 20

print("Bắt đầu quá trình huấn luyện...\n")

for epoch in range(EPOCHS):
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()

        outputs = model(batch_X)

        loss = criterion(outputs, batch_y)
        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        _, predicted_classes = torch.max(outputs, 1)
        correct_predictions += (predicted_classes == batch_y).sum().item()
        total_samples += batch_y.size(0)

    if (epoch + 1) % 2 == 0 or epoch == 0:
        epoch_loss = total_loss / len(dataloader)
        epoch_acc = (correct_predictions / total_samples) * 100
        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Mức độ sai sót (Loss): {epoch_loss:.4f} | Độ chính xác (Acc): {epoch_acc:.2f}%")

print("\nHoàn tất huấn luyện!")

Bắt đầu quá trình huấn luyện...

Epoch [01/20] | Mức độ sai sót (Loss): 0.4779 | Độ chính xác (Acc): 88.58%
Epoch [02/20] | Mức độ sai sót (Loss): 0.0201 | Độ chính xác (Acc): 99.53%
Epoch [04/20] | Mức độ sai sót (Loss): 0.0012 | Độ chính xác (Acc): 100.00%
Epoch [06/20] | Mức độ sai sót (Loss): 0.0004 | Độ chính xác (Acc): 100.00%
Epoch [08/20] | Mức độ sai sót (Loss): 0.0002 | Độ chính xác (Acc): 100.00%
Epoch [10/20] | Mức độ sai sót (Loss): 0.0001 | Độ chính xác (Acc): 100.00%
Epoch [12/20] | Mức độ sai sót (Loss): 0.0001 | Độ chính xác (Acc): 100.00%
Epoch [14/20] | Mức độ sai sót (Loss): 0.0001 | Độ chính xác (Acc): 100.00%
Epoch [16/20] | Mức độ sai sót (Loss): 0.0001 | Độ chính xác (Acc): 100.00%
Epoch [18/20] | Mức độ sai sót (Loss): 0.0000 | Độ chính xác (Acc): 100.00%
Epoch [20/20] | Mức độ sai sót (Loss): 0.0000 | Độ chính xác (Acc): 100.00%

Hoàn tất huấn luyện!


In [32]:
test_data = fetch_20newsgroups(subset='test', categories=categories, shuffle=True, random_state=42)

X_test = vectorizer.transform(test_data.data).toarray()
y_test = test_data.target

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)


model.eval()
with torch.no_grad():

    test_outputs = model(X_test_tensor)
    _, test_predicted = torch.max(test_outputs, 1)

    correct = (test_predicted == y_test_tensor).sum().item()
    accuracy = (correct / y_test_tensor.size(0)) * 100
    print(f"accuracy: {accuracy:.2f}%\n")


def predict_topic(sentence, model, vectorizer, target_names):
    X_new = vectorizer.transform([sentence]).toarray()
    X_tensor = torch.tensor(X_new, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        probabilities = model(X_tensor, training_mode=False)

        confidence, predicted_idx = torch.max(probabilities, 1)

    class_name = target_names[predicted_idx.item()]
    confidence_score = confidence.item() * 100

    return class_name, confidence_score

my_sentences = [
    "I love fast cars and driving on the highway.",
    "NASA launched a new rocket to the moon.",
    "The doctor prescribed me some antibiotics for the infection.",
    "I need to buy a new Nvidia GPU for rendering 3D graphics."
]

print("--- KẾT QUẢ DỰ ĐOÁN (INFERENCE) ---")
target_names = train_data.target_names

for text in my_sentences:
    topic, conf = predict_topic(text, model, vectorizer, target_names)
    print(f"Câu: '{text}'")
    print(f"-> Dự đoán: {topic.upper()} (Tự tin: {conf:.2f}%)\n")

accuracy: 93.14%

--- KẾT QUẢ DỰ ĐOÁN (INFERENCE) ---
Câu: 'I love fast cars and driving on the highway.'
-> Dự đoán: REC.AUTOS (Tự tin: 99.99%)

Câu: 'NASA launched a new rocket to the moon.'
-> Dự đoán: SCI.SPACE (Tự tin: 100.00%)

Câu: 'The doctor prescribed me some antibiotics for the infection.'
-> Dự đoán: SCI.MED (Tự tin: 100.00%)

Câu: 'I need to buy a new Nvidia GPU for rendering 3D graphics.'
-> Dự đoán: COMP.GRAPHICS (Tự tin: 100.00%)

